In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error as RMSE
import numpy as np
from category_encoders import TargetEncoder
from sklearn.linear_model import RidgeCV
path = Path("./data")


train_df = pd.read_csv(path / "train.csv")
test_df = pd.read_csv(path / "test.csv")

train_df["event_time"] = pd.to_datetime(train_df["event_time"], utc=True)
test_df["event_time"] = pd.to_datetime(test_df["event_time"], utc=True)

train_df["event_time"] = pd.to_datetime(train_df["event_time"])

train_df["product_id"] = train_df["product_id"].str.extract(r'(\d+)$').astype("Int64")
train_df["user_id"]    = train_df["user_id"].str.extract(r'(\d+)$').astype("Int64")
train_df["category_id"] = train_df["category_id"].str.extract(r'(\d+)$').astype("Int64")

test_df["product_id"] = test_df["product_id"].str.extract(r'(\d+)$').astype("Int64")
test_df["user_id"]    = test_df["user_id"].str.extract(r'(\d+)$').astype("Int64")
test_df["category_id"] = test_df["category_id"].str.extract(r'(\d+)$').astype("Int64")

In [2]:
def winsorize_series(s, lower=0.01, upper=0.99):
    lo = s.quantile(lower)
    hi = s.quantile(upper)
    return s.clip(lower=lo, upper=hi)

global_max = train_df["event_time"].max()
def transform(x_tr: pd.DataFrame, x_va: pd.DataFrame, x_te: pd.DataFrame):
    frames = (x_tr, x_va, x_te)
    eps = 1e-9
    
    for df in frames:
        mapping = {"ADD_CART": 0, "BUY": 1, "REMOVE_CART": 2, "VIEW": 3}
        df["event_type"] = df["event_type"].map(mapping).astype(int)
        
        df["events_in_session"] = np.log1p(
            df.groupby("user_session")["event_type"].transform("size")
        )
        df["events_per_user"] = np.log1p(
            df.groupby("user_id")["event_type"].transform("size")
        )
        df["add_per_user"] = np.log1p(
            df.groupby("user_id")["event_type"].transform(lambda x: (x == 0).sum())
        )
        df["buy_per_user"] = np.log1p(
            df.groupby("user_id")["event_type"].transform(lambda x: (x == 1).sum())
        )
        df["rm_per_user"] = np.log1p(
            df.groupby("user_id")["event_type"].transform(lambda x: (x == 2).sum())
        )
        df["view_per_user"] = np.log1p(
            df.groupby("user_id")["event_type"].transform(lambda x: (x == 3).sum())
        )
        df["add_in_session"] = np.log1p(
            df.groupby("user_session")["event_type"].transform(lambda x: (x == 0).sum())
        )
        df["buy_in_session"] = np.log1p(
            df.groupby("user_session")["event_type"].transform(lambda x: (x == 1).sum())
        )
        df["rm_in_session"] = np.log1p(
            df.groupby("user_session")["event_type"].transform(lambda x: (x == 2).sum())
        )
        df["view_in_session"] = np.log1p(
            df.groupby("user_session")["event_type"].transform(lambda x: (x == 3).sum())
        )

    for df in frames:
        first_t = df.groupby("user_session")["event_time"].transform("min")
        last_t  = df.groupby("user_session")["event_time"].transform("max")
        df["event_duration"] = (last_t - first_t).dt.total_seconds().abs().fillna(0.0)

    for df in frames:
        df["product_pop_users"]    = df.groupby("product_id")["user_id"].transform("nunique")
        df["product_pop_sessions"] = df.groupby("product_id")["user_session"].transform("nunique")
        df["prod_per_session"]  = df.groupby("user_session")["product_id"].transform("nunique")
        df["prod_per_user"]     = df.groupby("user_id")["product_id"].transform("nunique")
        df["sessions_per_user"] = df.groupby("user_id")["user_session"].transform("nunique")

    for df in frames:
        df["add_rate_user"]   = df["add_per_user"]   / (df["events_per_user"] + eps)
        df["buy_rate_user"]   = df["buy_per_user"]   / (df["events_per_user"] + eps)
        df["rm_rate_user"]    = df["rm_per_user"]    / (df["events_per_user"] + eps)
        df["view_rate_user"]  = df["view_per_user"]  / (df["events_per_user"] + eps)

        df["add_rate_session"]  = df["add_in_session"]  / (df["events_in_session"] + eps)
        df["buy_rate_session"]  = df["buy_in_session"]  / (df["events_in_session"] + eps)
        df["rm_rate_session"]   = df["rm_in_session"]   / (df["events_in_session"] + eps)
        df["view_rate_session"] = df["view_in_session"] / (df["events_in_session"] + eps)

        df["session_density"] = df["event_duration"] / (df["events_in_session"] + eps)

    for df in frames:
        last_sess_time = df.groupby("user_session")["event_time"].transform("max")
        recency_days = (global_max - last_sess_time).dt.total_seconds() / 86400.0
        df["recency_days"] = recency_days.fillna(recency_days.median())
        df["recency_inv"]  = 1.0 / (df["recency_days"] + 1.0)

    w_buy, w_add, w_rm = 0.7, 0.2, 0.4
    for df in frames:
        df["stability"] = (w_buy*df["buy_rate_user"] + w_add*df["add_rate_user"] - w_rm*df["rm_rate_user"]).clip(-1,1)

    for df in frames:
        s_log = winsorize_series(np.log1p(df["sessions_per_user"]), 0.01, 0.99)
        r_inv = df["recency_inv"].clip(0,1)
        stab  = df["stability"]
        raw   = 0.35*r_inv + 0.25*s_log + 0.40*stab
        lo, hi = raw.quantile(0.01), raw.quantile(0.99)
        #df["loyalty_score"] = ((raw.clip(lo,hi) - lo) / (hi - lo + 1e-9) * 100).round(2)

    cols_heavy = [
        "add_per_user","buy_per_user","rm_per_user","view_per_user",
        "product_pop_users","product_pop_sessions","prod_per_user","prod_per_session",
        "events_per_user","events_in_session","event_duration", "sessions_per_user", 
        "session_density"
    ]
    for df in frames:
        for c in cols_heavy:
            df[c + "_log1p"] = winsorize_series(np.log1p(df[c]), 0.01, 0.99)
        df.drop(columns=cols_heavy, inplace=True, errors="ignore")

    for df in frames:
        num_cols = df.select_dtypes(include=[np.number]).columns
        df[num_cols] = df[num_cols].replace([np.inf,-np.inf], np.nan)
        df[num_cols] = df[num_cols].fillna(df[num_cols].median())

    return x_tr, x_va, x_te

In [3]:
def build_session_value(df: pd.DataFrame) -> pd.Series:
    if "session_value" in df.columns:
        return df.groupby("user_session")["session_value"].max()
    return pd.Series(dtype=float)

def to_session_level(df: pd.DataFrame, target_name="session_value", is_train=True):
    last_idx = df.groupby("user_session")["event_time"].idxmax()
    ses_df = df.loc[last_idx].copy()
    if target_name in ses_df.columns:
        ses_df.drop(columns=[target_name], inplace=True)
    if is_train:
        ses_val = build_session_value(df)
        if not ses_val.empty:
            ses_df = ses_df.merge(
                ses_val.rename(target_name),
                left_on="user_session", right_index=True, how="left"
            )
    return ses_df


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, Dataset, DataLoader 
from skorch import NeuralNetRegressor
from skorch.dataset import ValidSplit 
from skorch.callbacks import EarlyStopping, LRScheduler, EpochScoring
import random
torch.manual_seed(42); np.random.seed(42); random.seed(42)
torch.cuda.manual_seed_all(42); torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
device = "cuda" if torch.cuda.is_available() else "cpu"

class FFNN(nn.Module):
    def __init__(self, in_dim, p=0.15): 
        super().__init__()
        self.in_norm = nn.LayerNorm(in_dim)

        self.fc1 = nn.Linear(in_dim, 64)
        self.s1 = nn.SiLU()
        self.bn1 = nn.LayerNorm(64)
        
        self.fc2 = nn.Linear(64, 128)
        self.s2 = nn.SiLU()
        self.bn2 = nn.LayerNorm(128)
        
        self.fc3 = nn.Linear(128+64, 128)
        self.s3 = nn.SiLU()
        self.bn3 = nn.LayerNorm(128)
                        
        self.fc4 = nn.Linear(128,1)

        self.d = nn.Dropout(p)

    def forward(self, x):
        x = self.in_norm(x)

        res = self.fc1(x)
        x = self.d(self.bn1(self.s1(res)))
        x = self.d(self.bn2(self.s2(self.fc2(x)))  )   
        x = torch.cat([x, res], dim=1)
        x = self.d(self.bn3(self.s3(self.fc3(x))))

        out = self.fc4(x)
        return out.squeeze(-1)  

In [5]:
from sklearn.model_selection import RepeatedKFold
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor, callback
import xgboost as xgb

all_sessions = train_df["user_session"].unique()
rkf = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)
test_session_ids = None

In [6]:
oof_list = []            
test_preds_lgb = []      
test_preds_ffn = []
test_preds_xgb = []

test_session_ids = None

rkf = RepeatedKFold(n_splits=5, n_repeats=1, random_state=42)

all_users = train_df["user_id"].unique()
for fold, (tr_idx, va_idx) in enumerate(rkf.split(all_users), 1):
    print(f"\n====== Fold {fold} ======")

    tr_users = all_users[tr_idx]
    va_users = all_users[va_idx]

    tr_rows = train_df[train_df["user_id"].isin(tr_users)].copy()
    va_rows = train_df[train_df["user_id"].isin(va_users)].copy()

    tr_rows, va_rows, te_rows = transform(tr_rows, va_rows, test_df.copy())

    tr_ses_df = to_session_level(tr_rows, target_name="session_value", is_train=True)
    va_ses_df = to_session_level(va_rows, target_name="session_value", is_train=True)
    te_ses_df = to_session_level(te_rows,  target_name="session_value", is_train=False)

    if test_session_ids is None:
        test_session_ids = te_ses_df["user_session"].values

    for df in (tr_ses_df, va_ses_df, te_ses_df):
        obj_cols = df.select_dtypes(include=['object']).columns.tolist()
        df.drop(columns=[c for c in obj_cols if c not in ['user_session','session_value']], inplace=True, errors='ignore')
        num_cols = df.select_dtypes(include=[np.number]).columns
        # Replace inf/-inf with NaN, then fill NaN with 0 (or use another strategy)
        df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)
        df[num_cols] = df[num_cols].fillna(0)
        df[num_cols] = df[num_cols].astype(np.float32)

    drop_cols = ["event_time","product_id","category_id","category_code",
                 "brand","user_id","user_session","session_value",
                 "session_value_y","session_value_x", "buy_rate_user", "view_in_session",
                 "buy_per_user_log1p","add_per_user_log1p","product_pop_users_log1p",
                 "product_pop_sessions_log1p","rm_per_user_log1p", "loyalty_score",
                 "events_per_user_log1p", "sessions_per_user_log1p", "stability", "view_per_user_log1p"]
    
    feature_cols = [c for c in tr_ses_df.columns if c not in drop_cols]

    X_tr = tr_ses_df[feature_cols]
    y_tr_log = np.log1p(tr_ses_df["session_value"].astype(np.float32))
    X_va = va_ses_df[feature_cols]
    y_va_log = np.log1p(va_ses_df["session_value"].astype(np.float32))
    y_va_raw = np.expm1(y_va_log)

    # -------------------- LGBM --------------------
    lgbm = LGBMRegressor(
        n_estimators=5000,
        learning_rate=0.02,
        num_leaves=127,
        subsample=0.9,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        min_gain_to_split=0.0,
        min_data_in_leaf=20,
        feature_pre_filter=False,
        objective="regression",
        random_state=42,
        n_jobs=-1,
        device="gpu" 
    )
    lgbm.fit(
        X_tr, y_tr_log,
        eval_set=[(X_va, y_va_log)],
        eval_metric="rmse",
        callbacks=[lgb.early_stopping(300), lgb.log_evaluation(200)]
    )

    va_pred_lgb_raw  = np.expm1(lgbm.predict(X_va, num_iteration=lgbm.best_iteration_))
    te_pred_lgb_raw  = np.expm1(lgbm.predict(te_ses_df[feature_cols]))

    print(f"[LGBM]  RMSE(va): {RMSE(y_va_raw, va_pred_lgb_raw):.4f} | R2(va): {r2_score(y_va_raw, va_pred_lgb_raw):.4f}")
    # -------------------- XGBoost --------------------
    dtrain = xgb.DMatrix(X_tr, label=y_tr_log)
    dvalid = xgb.DMatrix(X_va, label=y_va_log)
    dtest  = xgb.DMatrix(te_ses_df[feature_cols].astype(np.float32))


    params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",   
    "device": "cuda",            
    "learning_rate": 0.02,
    "max_depth": 8,
    "subsample": 0.9,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,
    "min_child_weight": 20,
    "seed": 42,
    }

    evals = [(dtrain, "train"), (dvalid, "valid")]

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=5000,
        evals=evals,
        early_stopping_rounds=300,
        verbose_eval=200
    )

    va_pred_xgb_raw = np.expm1(model.predict(dvalid, iteration_range=(0, model.best_iteration+1)))
    te_pred_xgb_raw = np.expm1(model.predict(dtest, iteration_range=(0, model.best_iteration+1)))

    print(f"[XGB] RMSE(va): {RMSE(y_va_raw, va_pred_xgb_raw):.4f} | R2(va): {r2_score(y_va_raw, va_pred_xgb_raw):.4f}")

    test_preds_xgb.append(te_pred_xgb_raw)
    
    # -------------------- FFNN --------------------
    scaler = StandardScaler()
    scaler.fit(X_tr.values)
    X_tr_np = scaler.transform(X_tr.values).astype(np.float32)
    X_va_np = scaler.transform(X_va.values).astype(np.float32)
    X_te_np = scaler.transform(te_ses_df[feature_cols].values).astype(np.float32)

    r2_cb = EpochScoring(scoring="r2", lower_is_better=False, on_train=False, name="valid_r2")

    net = NeuralNetRegressor(
        module=FFNN,                     
        module__in_dim=X_tr_np.shape[1], # giriş boyutu
        max_epochs=150,
        batch_size=256,
        optimizer=optim.AdamW,
        optimizer__lr=1e-3,
        optimizer__weight_decay=5e-4,
        criterion=nn.MSELoss(),
        train_split=ValidSplit(0.1, stratified=False),
        device=device,
        callbacks=[
            r2_cb,
            EarlyStopping(monitor='valid_loss', patience=15, load_best=True),
            LRScheduler(policy=optim.lr_scheduler.CosineAnnealingWarmRestarts,
                        T_0=10, T_mult=2, eta_min=1e-5),
        ],
        verbose=0
    )
    net.fit(X_tr_np, y_tr_log.to_numpy(dtype=np.float32))

    va_pred_ffn_raw = np.expm1(net.predict(X_va_np).reshape(-1))
    te_pred_ffn_raw = np.expm1(net.predict(X_te_np).reshape(-1))

    print(f"[FFNN] RMSE(va): {RMSE(y_va_raw, va_pred_ffn_raw):.4f} | R2(va): {r2_score(y_va_raw, va_pred_ffn_raw):.4f}")

    tmp = va_ses_df[["user_session"]].copy()
    tmp["y_true"]   = y_va_raw
    tmp["pred_lgb"] = va_pred_lgb_raw
    tmp["pred_ffn"] = va_pred_ffn_raw
    tmp["pred_xgb"] = va_pred_xgb_raw
    oof_list.append(tmp)

    test_preds_lgb.append(te_pred_lgb_raw)
    test_preds_ffn.append(te_pred_ffn_raw)


====== Fold 1 ======
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_gain_to_split is set=0.0, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.0
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_gain_to_split is set=0.0, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.0
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 2448
[LightGBM] [Info] Number of data points in the train set: 56740, number of used features: 18
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 4060 Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 9 dense feature groups (0.65 MB) transferre

In [7]:
oof_df = pd.concat(oof_list, ignore_index=True)

print("\n=== OOF (model bazında) ===")
print("LGBM  -> RMSE:", RMSE(oof_df["y_true"], oof_df["pred_lgb"]),
      " MAE:", mean_absolute_error(oof_df["y_true"], oof_df["pred_lgb"]),
      " R2:", r2_score(oof_df["y_true"], oof_df["pred_lgb"]))
print("FFNN  -> RMSE:", RMSE(oof_df["y_true"], oof_df["pred_ffn"]),
      " MAE:", mean_absolute_error(oof_df["y_true"], oof_df["pred_ffn"]),
      " R2:", r2_score(oof_df["y_true"], oof_df["pred_ffn"]))
print("\n=== OOF (XGB) ===")
print("RMSE:", RMSE(oof_df["y_true"], oof_df["pred_xgb"]),
      " MAE:", mean_absolute_error(oof_df["y_true"], oof_df["pred_xgb"]),
      " R2:", r2_score(oof_df["y_true"], oof_df["pred_xgb"]))

xgb_test_mean = np.mean(np.vstack(test_preds_xgb), axis=0)
lgb_test_mean  = np.mean(np.vstack(test_preds_lgb), axis=0)
ffnn_test_mean = np.mean(np.vstack(test_preds_ffn), axis=0)


=== OOF (model bazında) ===
LGBM  -> RMSE: 19.89031496657405  MAE: 11.092626950451276  R2: 0.8254547396914109
FFNN  -> RMSE: 22.148574829101562  MAE: 11.910734176635742  R2: 0.7835705876350403

=== OOF (XGB) ===
RMSE: 19.64943504333496  MAE: 11.055890083312988  R2: 0.8296567797660828


In [9]:
X_meta_train = np.log1p(oof_df[["pred_lgb","pred_ffn", "pred_xgb"]].values)
y_meta_train = np.log1p(oof_df["y_true"].values)

X_meta_test  = np.log1p(np.column_stack([lgb_test_mean, ffnn_test_mean, xgb_test_mean]))

meta = RidgeCV(alphas=np.logspace(-4, 2, 25), cv=10)
meta.fit(X_meta_train, y_meta_train)

oof_meta_log  = meta.predict(X_meta_train)
test_meta_log = meta.predict(X_meta_test)

oof_meta  = np.expm1(oof_meta_log)
test_meta = np.expm1(test_meta_log)

print("\n=== OOF (META) ===")
print("META -> RMSE:", RMSE(oof_df["y_true"], oof_meta),
      " MAE:", mean_absolute_error(oof_df["y_true"], oof_meta),
      " R2:", r2_score(oof_df["y_true"], oof_meta))

sub = pd.DataFrame({"user_session": test_session_ids, "session_value": test_meta})
sub.to_csv("submission_stacking.csv", index=False)
print("Saved: submission_stacking.csv")


=== OOF (META) ===
META -> RMSE: 19.63759063299068  MAE: 11.027650731939382  R2: 0.829862070033557
Saved: submission_stacking.csv


In [10]:
from sklearn.inspection import permutation_importance

r = permutation_importance(
    lgbm, X_va, y_va_log,
    n_repeats=30,
    n_jobs=-1, random_state=42
)

[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_gain_to_split is set=0.0, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.0
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_gain_to_split is set=0.0, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.0
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_gain_to_split is set=0.0, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.0
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] min_gain_to_split is set=0.0, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.0
[LightGBM] [Warn

In [11]:
for i in r.importances_mean.argsort()[::-1]:
    if r.importances_mean[i] - 2 * r.importances_std[i] > 0:
        print(f"{X_va.columns[i]:<20}"
              f"{r.importances_mean[i]:.3f}"
              f" +/- {r.importances_std[i]:.3f}")

buy_in_session      0.506 +/- 0.007
add_in_session      0.199 +/- 0.003
view_rate_session   0.074 +/- 0.002
prod_per_user_log1p 0.045 +/- 0.002
buy_rate_session    0.035 +/- 0.001
recency_inv         0.033 +/- 0.001
recency_days        0.033 +/- 0.001
prod_per_session_log1p0.017 +/- 0.001
view_rate_user      0.016 +/- 0.001
rm_rate_session     0.013 +/- 0.001
add_rate_session    0.012 +/- 0.001
event_duration_log1p0.011 +/- 0.001
add_rate_user       0.011 +/- 0.001
rm_rate_user        0.006 +/- 0.001
events_in_session_log1p0.005 +/- 0.001
session_density_log1p0.003 +/- 0.000
event_type          0.001 +/- 0.000
rm_in_session       0.001 +/- 0.000


In [12]:
r_ = permutation_importance(
    net, X_va_np, y_va_log,
    n_repeats=10,
    n_jobs=1, random_state=42
)

In [13]:
feature_names = X_va.columns
for i in r_.importances_mean.argsort()[::-1]:
    if r_.importances_mean[i] - 2 * r_.importances_std[i] > 0:
        print(f"{feature_names[i]:<20}"
              f"{r_.importances_mean[i]:.3f}"
              f" +/- {r_.importances_std[i]:.3f}")


buy_in_session      0.566 +/- 0.007
rm_rate_session     0.324 +/- 0.005
add_in_session      0.247 +/- 0.004
add_rate_session    0.183 +/- 0.004
buy_rate_session    0.107 +/- 0.002
view_rate_user      0.081 +/- 0.002
prod_per_user_log1p 0.081 +/- 0.002
add_rate_user       0.071 +/- 0.002
recency_days        0.070 +/- 0.002
recency_inv         0.067 +/- 0.002
view_rate_session   0.059 +/- 0.001
rm_in_session       0.055 +/- 0.002
events_in_session_log1p0.054 +/- 0.001
prod_per_session_log1p0.046 +/- 0.002
session_density_log1p0.035 +/- 0.001
rm_rate_user        0.026 +/- 0.001
event_duration_log1p0.019 +/- 0.001
event_type          0.014 +/- 0.001


In [20]:
class XGBWrapper:
    def __init__(self, booster):
        self.booster = booster
    def fit(self, X, y=None):
        return self   # sklearn interface için dummy
    def predict(self, X):
        dm = xgb.DMatrix(X)
        return self.booster.predict(dm)

wrapped = XGBWrapper(model)

r__ = permutation_importance(
    wrapped, X_va, y_va_log,
    n_repeats=10,
    n_jobs=1,  # multiprocessing sorun çıkarmasın
    random_state=42,
    scoring="neg_root_mean_squared_error"
)

for i in r__.importances_mean.argsort()[::-1]:
    if r__.importances_mean[i] - 2 * r__.importances_std[i] > 0:
        print(f"{X_va.columns[i]:<20}"
              f"{r__.importances_mean[i]:.3f}"
              f" +/- {r__.importances_std[i]:.3f}")


buy_in_session      0.300 +/- 0.003
add_in_session      0.093 +/- 0.001
prod_per_user_log1p 0.033 +/- 0.001
view_rate_session   0.029 +/- 0.001
recency_days        0.027 +/- 0.001
view_rate_user      0.019 +/- 0.000
recency_inv         0.012 +/- 0.001
add_rate_user       0.012 +/- 0.001
add_rate_session    0.012 +/- 0.000
rm_rate_session     0.011 +/- 0.001
prod_per_session_log1p0.011 +/- 0.000
buy_rate_session    0.008 +/- 0.000
event_duration_log1p0.008 +/- 0.000
rm_rate_user        0.007 +/- 0.000
session_density_log1p0.005 +/- 0.000
events_in_session_log1p0.003 +/- 0.000
rm_in_session       0.001 +/- 0.000
event_type          0.000 +/- 0.000
